in this file we will downsample our tiff files 

In [1]:
import os
import rasterio
from rasterio.enums import Resampling
import numpy as np

In [2]:
def find_all_tiff_files(root_folder):
    tiff_files = []
    for root, dirs, files in os.walk(root_folder):
        for file in files:
            if file.lower().endswith(('.tif', '.tiff')):
                tiff_files.append(os.path.join(root, file))
    return tiff_files

In [3]:
all_tiffs = find_all_tiff_files("../../Resampled/")
for i in all_tiffs : 
        print(i)
        with rasterio.open(i) as src:
                src_res_x, src_res_y = abs(src.transform.a), abs(src.transform.e)
                print(src_res_x, src_res_y)

../../Resampled/climat\wc2.1_30s_prec_01.tif
0.004491578290704928 0.004491578290704928
../../Resampled/climat\wc2.1_30s_srad_01.tif
0.004491578290704928 0.004491578290704928
../../Resampled/climat\wc2.1_30s_tmax_01.tif
0.004491578290704928 0.004491578290704928
../../Resampled/climat\wc2.1_30s_tmin_01.tif
0.004491578290704928 0.004491578290704928
../../Resampled/climat\wc2.1_30s_vapr_01.tif
0.004491578290704928 0.004491578290704928
../../Resampled/climat\wc2.1_30s_wind_01.tif
0.004491578290704928 0.004491578290704928
../../Resampled/organisme\amphibians.tif
30000.0 30000.0
../../Resampled/organisme\barren.tif
0.004696795741571862 0.004491616674357792
../../Resampled/organisme\birds.tif
30000.0 30000.0
../../Resampled/organisme\cultivated.tif
0.004696795741571862 0.004491616674357792
../../Resampled/organisme\deciduousTrees.tif
0.004696795741571862 0.004491616674357792
../../Resampled/organisme\evergreen.tif
0.004696795741571862 0.004491616674357792
../../Resampled/organisme\herbaceous.t

In [4]:
import os
import rasterio
from rasterio.enums import Resampling
from rasterio.warp import calculate_default_transform, reproject
import numpy as np

TARGET_RESOLUTION = (0.008333333300000002 ,0.008333333300000002)
CATEGORICAL_FILES = {
    "amphibians.tif",
    "birds.tif",
    "mammals.tif",
    "reptiles.tif",
    "human_activities.tif",
    "barren.tif",
    "cultivated.tif",
    "deciduousTrees.tif",
    "evergreen.tif",
    "herbaceous.tif",
    "openWater.tif",
    "RegFlooded.tif",
    "shrubs.tif",
    "snow&ice.tif",
    "trees.tif",
    "urban.tif",
    "rocksType_1km.tif",
    "40N_000E.tif",
    "n00e035_hnd.tif",
    "n00e040_hnd.tif",
    "n05e035_hnd.tif",
    "n05e040_hnd.tif"
}

def is_categorical(file_path):
    return os.path.basename(file_path) in CATEGORICAL_FILES

def normalize_crs(src, target_crs='EPSG:4326'):
    if src.crs.to_string() == target_crs:
        return src
    else:
        print(f"Reprojecting {src.name} from {src.crs} to {target_crs}")
        transform, width, height = calculate_default_transform(
            src.crs, target_crs, src.width, src.height, *src.bounds
        )

        kwargs = src.meta.copy()
        kwargs.update({
            'crs': target_crs,
            'transform': transform,
            'width': width,
            'height': height
        })

        memfile = rasterio.io.MemoryFile()
        dst = memfile.open(**kwargs)

        for i in range(1, src.count + 1):
            reproject(
                source=rasterio.band(src, i),
                destination=rasterio.band(dst, i),
                src_transform=src.transform,
                src_crs=src.crs,
                dst_transform=transform,
                dst_crs=target_crs,
                resampling=Resampling.nearest if is_categorical(src.name) else Resampling.bilinear
            )
        return dst

def resample_to_fixed_resolution(tif_path, output_folder, target_resolution=TARGET_RESOLUTION):
    with rasterio.open(tif_path) as src_orig:
        src = normalize_crs(src_orig)
        src_res_x, src_res_y = abs(src.transform.a), abs(src.transform.e)

        if np.isclose(src_res_x, target_resolution[0], rtol=1e-6) and np.isclose(src_res_y, target_resolution[1], rtol=1e-6):
            print(f"Already at target resolution: {tif_path}")
            return

        scale_x = src_res_x / target_resolution[0]
        scale_y = src_res_y / target_resolution[1]

        new_width = int(src.width * scale_x)
        new_height = int(src.height * scale_y)

        resample_method = Resampling.nearest if is_categorical(tif_path) else Resampling.bilinear
        try:
            data = src.read(
                out_shape=(src.count, new_height, new_width),
                resampling=resample_method
            )

            new_transform = src.transform * src.transform.scale(
                src.width / new_width,
                src.height / new_height
            )

            profile = src.profile
            profile.update({
                'transform': new_transform,
                'height': new_height,
                'width': new_width
            })

            os.makedirs(output_folder, exist_ok=True)
            filename = os.path.basename(tif_path)
            output_path = os.path.join(output_folder, filename)

            with rasterio.open(output_path, 'w', **profile) as dst:
                dst.write(data)

            print(f"✅ Resampled and saved: {output_path}")
        except Exception as e:
            print(tif_path)
            print(e)

def batch_resample_to_fixed_resolution(file_list, output_folder):
    print(f"Target resolution: {TARGET_RESOLUTION}")
    for fp in file_list:
        resample_to_fixed_resolution(fp, output_folder)

In [13]:
"""all_tiffs = find_all_tiff_files("../v1/")
output_folder = '../Resampled/'
batch_resample_to_fixed_resolution(all_tiffs, output_folder)"""

'all_tiffs = find_all_tiff_files("../v1/")\noutput_folder = \'../Resampled/\'\nbatch_resample_to_fixed_resolution(all_tiffs, output_folder)'

In [6]:
batch_resample_to_fixed_resolution(find_all_tiff_files('../../Resampled/'), "../Resampled2/")

Target resolution: (0.008333333300000002, 0.008333333300000002)
✅ Resampled and saved: ../Resampled2/wc2.1_30s_prec_01.tif
✅ Resampled and saved: ../Resampled2/wc2.1_30s_srad_01.tif
✅ Resampled and saved: ../Resampled2/wc2.1_30s_tmax_01.tif
✅ Resampled and saved: ../Resampled2/wc2.1_30s_tmin_01.tif
✅ Resampled and saved: ../Resampled2/wc2.1_30s_vapr_01.tif
✅ Resampled and saved: ../Resampled2/wc2.1_30s_wind_01.tif
Reprojecting ../../Resampled/organisme\amphibians.tif from ESRI:54009 to EPSG:4326
✅ Resampled and saved: ../Resampled2/amphibians.tif
✅ Resampled and saved: ../Resampled2/barren.tif
Reprojecting ../../Resampled/organisme\birds.tif from ESRI:54009 to EPSG:4326
✅ Resampled and saved: ../Resampled2/birds.tif
✅ Resampled and saved: ../Resampled2/cultivated.tif
✅ Resampled and saved: ../Resampled2/deciduousTrees.tif
✅ Resampled and saved: ../Resampled2/evergreen.tif
✅ Resampled and saved: ../Resampled2/herbaceous.tif
✅ Resampled and saved: ../Resampled2/mammals.tif
✅ Resampled an

In [7]:
batch_resample_to_fixed_resolution(["../../Resampled/human_activities.tif"], "../Resampled2/")

Target resolution: (0.008333333300000002, 0.008333333300000002)
✅ Resampled and saved: ../Resampled2/human_activities.tif


In [8]:
batch_resample_to_fixed_resolution(["../Resampled2/landuse.tif"], "../Resampled2/")

Target resolution: (0.008333333300000002, 0.008333333300000002)
../Resampled2/landuse.tif
Deleting ../Resampled2/landuse.tif failed: Permission denied


In [5]:
import rasterio
with rasterio.open("..\..\Resampled2\output_1km.tif") as src:
    data = src.read()  # shape: (bands, height, width)
    print(data.shape)


(1, 2174, 2479)
